In [ ]:
import os, sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])

pip_install([
    "numpy<2.0.0",
    "pyspark==3.5.1",
    "sentence-transformers",
    "pandas",
    "pyarrow>=16,<18"
])

subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jdk-headless"])
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T

spark = (
    SparkSession.builder
    .appName("cse488-m2")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")  
    .getOrCreate()
)

path = "/content/combined_laptops.csv" 
df = spark.read.option("header", True).option("multiLine", True).option("quote", '"').option("escape", '"').csv(path)
print(df.count(), "rows")
df.printSchema()

2022 rows
root
 |-- title: string (nullable = true)
 |-- price_usd: string (nullable = true)
 |-- price_original: string (nullable = true)
 |-- price_original_currency: string (nullable = true)
 |-- cpu: string (nullable = true)
 |-- ram_gb: string (nullable = true)
 |-- storage: string (nullable = true)
 |-- gpu: string (nullable = true)
 |-- display: string (nullable = true)
 |-- battery: string (nullable = true)
 |-- category: string (nullable = true)
 |-- document_text: string (nullable = true)
 |-- has_review_text: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_collection_date: string (nullable = true)
 |-- source_row_id: string (nullable = true)
 |-- has_page_provenance: string (nullable = true)



In [ ]:
import hashlib
from pyspark.sql import functions as F, types as T

@F.udf(T.StringType())
def stable_id(source_dataset, source_row_id):
    raw = f"{source_dataset}|{source_row_id}"
    return hashlib.md5(raw.encode()).hexdigest()

clean = (
    df
    .withColumn("price_usd", F.col("price_usd").cast("double"))
    .withColumn("price_original", F.col("price_original").cast("double"))
    .withColumn("ram_gb", F.col("ram_gb").cast("double"))
    .withColumn("title", F.trim(F.col("title")))
    .withColumn("document_text", F.trim(F.col("document_text")))
    .withColumn("battery", F.when(F.trim(F.col("battery")) == "", None).otherwise(F.col("battery")))
    .withColumn("gpu", F.when(F.trim(F.col("gpu")) == "", None).otherwise(F.col("gpu")))
    .filter(F.col("document_text").isNotNull() & (F.length("document_text") > 0))
    .filter(F.col("price_usd").isNotNull())
    .withColumn("row_uid", stable_id(F.col("source_dataset"), F.col("source_row_id")))  # deterministic, not monotonic
    .cache()  
)

clean.count()  
print("after cleaning:", clean.count())

after cleaning: 2022


In [ ]:
from pyspark.ml.feature import RegexTokenizer, HashingTF, MinHashLSH

tokenizer = RegexTokenizer(inputCol="document_text", outputCol="tokens", pattern=r"\W+", minTokenLength=2)
tokenized = tokenizer.transform(clean)

hashing_tf = HashingTF(inputCol="tokens", outputCol="tf_vec", numFeatures=1 << 14, binary=True)
featurized = hashing_tf.transform(tokenized)

mh = MinHashLSH(inputCol="tf_vec", outputCol="minhash", numHashTables=5)
mh_model = mh.fit(featurized)
hashed = mh_model.transform(featurized)

JACCARD_DIST_THRESHOLD = 0.2  
pairs = mh_model.approxSimilarityJoin(hashed, hashed, JACCARD_DIST_THRESHOLD, distCol="jaccard_dist") \
    .filter("datasetA.row_uid < datasetB.row_uid") \
    .select(
        F.col("datasetA.row_uid").alias("uid_a"),
        F.col("datasetB.row_uid").alias("uid_b"),
        "jaccard_dist",
    )

print("near-duplicate pairs found:", pairs.count())
pairs.orderBy("jaccard_dist").show(10, truncate=60)

dupe_to_keep = pairs.groupBy("uid_b").agg(F.min("uid_a").alias("keep_uid"))
to_drop = dupe_to_keep.select(F.col("uid_b").alias("row_uid"))

deduped = hashed.join(to_drop, on="row_uid", how="left_anti")
print("rows before dedup:", hashed.count(), " | after dedup:", deduped.count())

near-duplicate pairs found: 792
+--------------------------------+--------------------------------+------------+
|                           uid_a|                           uid_b|jaccard_dist|
+--------------------------------+--------------------------------+------------+
|0fd2106d53ebe91db4afe21729c14712|3e4794bb90a8fe2198e3e2dbfedbdd81|         0.0|
|39d6f02d79e29fce99a7d354dad12e71|bb8fcf8d9b5e4949331e77cf93f7adc2|         0.0|
|8d2f56c396dc220d7901f93c9e6c212d|95bda8cea4a3dcd1b0c3c55a42f225c2|         0.0|
|122b554ac394c93d8d4ac827aa2abba6|6e92f681a1550ca6d770699565f55efb|         0.0|
|3fd3ce1407dc0b9e92a671bea7b79580|c4cc63ac34ffa65fd465a8371e36b4e7|         0.0|
|39d6f02d79e29fce99a7d354dad12e71|a08f38fb7be016f1dcbc314ca45276e4|         0.0|
|9b298bcef46e25021005a702781cf06d|fb5102ec4ef60519d846420eaef3bdcf|         0.0|
|6e92f681a1550ca6d770699565f55efb|84439fed88001d9826fd7a3402801dd2|         0.0|
|1ec7e81939d343bd69360f51b306f9fe|c4cc63ac34ffa65fd465a8371e36b4e7|         0

In [ ]:
CHUNK_WORDS = 120
CHUNK_OVERLAP = 20

def chunk_partition(rows):
    for row in rows:
        text = row["document_text"] or ""
        words = text.split()
        if not words:
            continue
        start = 0
        idx = 0
        n = len(words)
        while start < n:
            end = min(start + CHUNK_WORDS, n)
            chunk_text = " ".join(words[start:end])
            yield (row["row_uid"], idx, chunk_text)
            if end == n:
                break
            start = end - CHUNK_OVERLAP
            idx += 1

chunk_schema = T.StructType([
    T.StructField("row_uid", T.StringType(), False),
    T.StructField("chunk_idx", T.IntegerType(), False),
    T.StructField("chunk_text", T.StringType(), False),
])

chunks_rdd = deduped.select("row_uid", "document_text").rdd.mapPartitions(chunk_partition)
chunks_df = spark.createDataFrame(chunks_rdd, schema=chunk_schema)

print("total chunks:", chunks_df.count())

metadata_cols = [
    "row_uid", "title", "price_usd", "price_original", "price_original_currency",
    "cpu", "ram_gb", "storage", "gpu", "display", "battery", "category",
    "has_review_text", "source_dataset", "source_collection_date",
    "source_row_id", "has_page_provenance",
]
enriched = chunks_df.join(deduped.select(*metadata_cols), on="row_uid", how="left")

total chunks: 2643


In [ ]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

EMBED_MODEL_NAME = "all-MiniLM-L6-v2" 

@pandas_udf(T.ArrayType(T.FloatType()))
def embed_batch(texts: pd.Series) -> pd.Series:
    from sentence_transformers import SentenceTransformer
    global _model
    if "_model" not in globals():
        _model = SentenceTransformer(EMBED_MODEL_NAME)
    vecs = _model.encode(texts.tolist(), batch_size=64, show_progress_bar=False)
    return pd.Series([v.tolist() for v in vecs])

embedded = enriched.coalesce(4).withColumn("embedding", embed_batch(F.col("chunk_text")))

embedded.select("row_uid", "chunk_idx", "chunk_text", "embedding").show(3, truncate=60)

+--------------------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|                         row_uid|chunk_idx|                                                  chunk_text|                                                   embedding|
+--------------------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|558389f6752bb2a5f024a76c8b26a6ed|        0|Product: ASUS Vivobook S 16" 3K OLED Intel Core Ultra 7 2...|[-0.07516633, -0.003686094, -0.07122591, -0.0111141605, 0...|
|558389f6752bb2a5f024a76c8b26a6ed|        1|13, Intel AI Boost NPU up to 13TOPS; Screen Size: 16"; To...|[-0.03809445, -0.017700203, 0.020603087, -0.030108975, 0....|
|558389f6752bb2a5f024a76c8b26a6ed|        2|40Gbps); HDMI: 1 x HDMI 2.1; Audio Ports: 1 x 3.5mm Combo...|[-0.008704036, -0.014219164, -0.008708177, -0.014658575, ...

In [ ]:
out_path = "laptop_chunks_embeddings.parquet"

(
    embedded
    .withColumn("chunk_id", F.concat_ws("_", F.col("row_uid"), F.col("chunk_idx")))
    .write
    .mode("overwrite")
    .parquet(out_path)
)

print("wrote", out_path)
result = spark.read.parquet(out_path)
print("rows:", result.count())
result.printSchema()

wrote laptop_chunks_embeddings.parquet
rows: 2643
root
 |-- row_uid: string (nullable = true)
 |-- chunk_idx: integer (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- price_original: double (nullable = true)
 |-- price_original_currency: string (nullable = true)
 |-- cpu: string (nullable = true)
 |-- ram_gb: double (nullable = true)
 |-- storage: string (nullable = true)
 |-- gpu: string (nullable = true)
 |-- display: string (nullable = true)
 |-- battery: string (nullable = true)
 |-- category: string (nullable = true)
 |-- has_review_text: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_collection_date: string (nullable = true)
 |-- source_row_id: string (nullable = true)
 |-- has_page_provenance: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- chunk_id: string (nullable = true)



In [ ]:
print("Original rows:", clean.count())
print("Rows after MinHash/LSH dedup:", deduped.count())
print("Duplicate rows removed:", clean.count() - deduped.count())
print("Total chunks generated:", chunks_df.count())
print("Avg chunks per device:", chunks_df.count() / deduped.count())
print("Embedding dim:", len(embedded.select("embedding").first()["embedding"]))

Original rows: 2022
Rows after MinHash/LSH dedup: 1503
Duplicate rows removed: 519
Total chunks generated: 2643
Avg chunks per device: 1.7584830339321358
Embedding dim: 384


In [ ]:
inspect = pairs.join(
        hashed.select(F.col("row_uid").alias("uid_a"), F.col("title").alias("title_a"), F.col("source_dataset").alias("src_a")),
        on="uid_a"
    ).join(
        hashed.select(F.col("row_uid").alias("uid_b"), F.col("title").alias("title_b"), F.col("source_dataset").alias("src_b")),
        on="uid_b"
    )

inspect.orderBy("jaccard_dist").select("title_a", "src_a", "title_b", "src_b", "jaccard_dist").show(30, truncate=40)

pairs_with_src = inspect.withColumn("same_source", F.col("src_a") == F.col("src_b"))
pairs_with_src.groupBy("same_source", "src_a", "src_b").count().show()

+----------------------------------------+-------------------+----------------------------------------+-------------------+------------+
|                                 title_a|              src_a|                                 title_b|              src_b|jaccard_dist|
+----------------------------------------+-------------------+----------------------------------------+-------------------+------------+
|  HP 15-AC110nv (i7-6500U/6GB/1TB/Radeon|kaggle_laptop_price|  HP 15-AC110nv (i7-6500U/6GB/1TB/Radeon|kaggle_laptop_price|         0.0|
|                      Dell Inspiron 3552|kaggle_laptop_price|                      Dell Inspiron 3552|kaggle_laptop_price|         0.0|
|                   Lenovo Yoga 900-13ISK|kaggle_laptop_price|                   Lenovo Yoga 900-13ISK|kaggle_laptop_price|         0.0|
|                      Dell Inspiron 3567|kaggle_laptop_price|                      Dell Inspiron 3567|kaggle_laptop_price|         0.0|
|                     Acer Aspire ES1-531

In [ ]:
dupe_price_check = inspect.join(
        hashed.select(F.col("row_uid").alias("uid_a"), F.col("price_usd").alias("price_a")),
        on="uid_a"
    ).join(
        hashed.select(F.col("row_uid").alias("uid_b"), F.col("price_usd").alias("price_b")),
        on="uid_b"
    ).filter(F.col("jaccard_dist") == 0.0)

dupe_price_check.withColumn("price_diff", F.abs(F.col("price_a") - F.col("price_b"))) \
    .agg(F.avg("price_diff").alias("avg_price_diff"), F.max("price_diff").alias("max_price_diff"), F.count("*").alias("n")) \
    .show()

dupe_price_check.withColumn("price_diff", F.abs(F.col("price_a") - F.col("price_b"))) \
    .orderBy(F.desc("price_diff")).select("uid_a", "price_a", "uid_b", "price_b", "price_diff").show(10)

+--------------+--------------+---+
|avg_price_diff|max_price_diff|  n|
+--------------+--------------+---+
|           0.0|           0.0| 43|
+--------------+--------------+---+

+--------------------+------------------+--------------------+------------------+----------+
|               uid_a|           price_a|               uid_b|           price_b|price_diff|
+--------------------+------------------+--------------------+------------------+----------+
|39d6f02d79e29fce9...| 878.5999999999999|a08f38fb7be016f1d...| 878.5999999999999|       0.0|
|122b554ac394c93d8...|435.84999999999997|6e92f681a1550ca6d...|435.84999999999997|       0.0|
|1f602caf097fae335...|           1723.85|deb5cca9a4fae94ef...|           1723.85|       0.0|
|7fd3952df1c334a46...| 527.8499999999999|8e7b6aeb3007979d8...| 527.8499999999999|       0.0|
|b7874c577dca654ea...|332.34999999999997|df9ee4894493796a1...|332.34999999999997|       0.0|
|6e92f681a1550ca6d...|435.84999999999997|84439fed88001d982...|435.849999999

In [ ]:
import shutil
shutil.make_archive("laptop_chunks_embeddings", "zip", "laptop_chunks_embeddings.parquet")

from google.colab import files
files.download("laptop_chunks_embeddings.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile
zipfile.ZipFile("laptop_chunks_embeddings.zip").extractall("laptop_chunks_embeddings.parquet")

import pandas as pd
df = pd.read_parquet("laptop_chunks_embeddings.parquet")
print(df.shape)

(2643, 21)


In [ ]:
import pandas as pd

df = pd.read_parquet("laptop_chunks_embeddings.parquet")  

df["lineage"] = (
    "chunk:" + df["chunk_id"].astype(str)
    + " <- device:" + df["row_uid"].astype(str)
    + " <- source_row:" + df["source_row_id"].astype(str)
    + " <- dataset:" + df["source_dataset"].astype(str)
)

df.to_parquet("laptop_chunks_embeddings_with_lineage.parquet", index=False)

print(df[["chunk_id", "row_uid", "source_row_id", "source_dataset", "lineage"]].head(3).to_string())

                             chunk_id                           row_uid                                                                                                                                                           source_row_id source_dataset                                                                                                                                                                                                                                                                                            lineage
0  558389f6752bb2a5f024a76c8b26a6ed_0  558389f6752bb2a5f024a76c8b26a6ed  https://www.newegg.com/asus-vivobook-16-2880x1800-oled-intel-core-ultra-7-255h-intel-arc-graphics-gpu-32-gb-memory-1-tb-pcie-g4-ssd-no-hdd-hdd-black/p/N82E16834236727  newegg_scrape  chunk:558389f6752bb2a5f024a76c8b26a6ed_0 <- device:558389f6752bb2a5f024a76c8b26a6ed <- source_row:https://www.newegg.com/asus-vivobook-16-2880x1800-oled-intel-core-ultra-7-255h-intel-arc-graphics-gp

In [ ]:
from google.colab import files
files.download("laptop_chunks_embeddings_with_lineage.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>